# Notebook 03) Data Cleaning and Feature Engineering

In [10]:
import sys
sys.path.append("..")
from utils import *

df = load_data()
print("Data loaded successfully:", df.shape)

Data loaded successfully: (119143, 28)


## Data Cleaning
In this section I clean the master dataframe by removing unnecessary columns, handling missing values, and converting date columns to the correct format to prepare 
the data for model building.

### Step 1) Drop Unnecessary Columns

In [11]:
# Drop columns that are not going to be use for the model since we can already see them on the dataset overview above 
df = df.drop(columns=[
    "review_comment_title",   # too many missing values
    "review_comment_message", # too many missing values
    "review_id",              # not a useful feature
    "order_item_id",          # not needed
    "product_id",             # not needed
    "seller_id",              # not needed
    "customer_unique_id",     # duplicate of customer_id
    "customer_zip_code_prefix" # too granular
])

print("Columns after dropping:", df.shape)

Columns after dropping: (119143, 20)


### Step 2) Drop Missing Values

In [12]:
# Drop rows with missing review scores since that is my target variable
df = df.dropna(subset=["review_score"])

# Drop remaining rows with missing values
df = df.dropna()

print("Shape after dropping missing values:", df.shape)

Shape after dropping missing values: (114842, 20)


### Step 3) Convert Date Columns to Datetime Format

In [13]:
# Convert date columns from string to datetime format
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])
print("Columns converted succesfully")

Columns converted succesfully


## Feature Engineering
In this section two new features are created from the existing date columns to capture delivery performance. These features are expected to be strong predictors of customer satisfaction since late or slow deliveries are likely to result in negative reviews.

In [14]:
# Calculate how many days it took to deliver the order
df["delivery_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

# Calculate if order was delivered late
df["delivered_late"] = (
    df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]
).astype(int)

print("New features created successfully")
print(df[["delivery_days", "delivered_late"]].head())

New features created successfully
   delivery_days  delivered_late
0              8               0
1              8               0
2              8               0
3             13               0
4              9               0


## Final Check  Confirm Clean Data

In [15]:
# Final check on the cleaned dataframe
print("Final shape:", df.shape)

# Confirm no missing values remain
print("\nMissing values remaining:")
print(df.isnull().sum())

Final shape: (114842, 22)

Missing values remaining:
order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
review_score                     0
review_creation_date             0
review_answer_timestamp          0
shipping_limit_date              0
price                            0
freight_value                    0
payment_sequential               0
payment_type                     0
payment_installments             0
payment_value                    0
customer_city                    0
customer_state                   0
delivery_days                    0
delivered_late                   0
dtype: int64


## Data Cleaning Summary
After removing unnecessary columns, dropping rows with missing values, converting date columns to datetime format, and engineering two new features, the cleaned dataframe is fully prepared for model building with zero missing values remaining.

### Note on Reusability
The cleaning and feature engineering steps in this notebook have been added to utils.py as a clean_data() function. 

This allows all subsequent notebooks to load and clean the data in one step by calling load_data() followed by clean_data(), ensuring consistency across the entire project pipeline.